[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MeiraDavidson/math_module2/blob/main/ch2_limits.ipynb)

# Chapter 2 — Companion Notebook
## Getting Arbitrarily Close
*What a function is doing right next to a point it might never reach*

Four live pictures of the chapter's one big idea — that a limit listens to what a function does *on the approach* and never cares what happens *at* the point itself: zoom into a hole, win the $\varepsilon$–$\delta$ game, run the Intermediate Value Theorem as an algorithm, and squeeze $\tfrac{\sin x}{x}$ to its surprising value.

> **How to use this notebook.** Run every cell from the top (Shift+Enter). Each widget below is live — drag the sliders and watch the picture respond. Requires `ipywidgets` and `matplotlib`.

In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from ipywidgets import (interact, interactive, fixed, Dropdown, Checkbox,
                        Button, Output, VBox, HBox,
                        FloatSlider, IntSlider, FloatLogSlider)
from IPython.display import HTML, display, Markdown

# Book 2 palette — matches the printed figures in figures/png/
BLUE="#1f4e79"; RED="#c0392b"; GREEN="#2e8b57"; ORANGE="#e08e0b"
PURPLE="#6c3483"; TEAL="#1b9e9e"; GRAY="#7f8c8d"; DARK="#2c3e50"
SHADE="#9fc5e8"; SHADE2="#f4b6ad"; SHADEG="#a9dfbf"; LIGHT="#d6dbdf"

plt.rcParams.update({
    "figure.figsize": (7.2, 4.5), "figure.dpi": 96,
    "font.family": "serif", "mathtext.fontset": "cm", "font.size": 12,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": LIGHT, "grid.linewidth": 0.7,
    "lines.linewidth": 2.2, "lines.solid_capstyle": "round",
})

def axes(ax=None, title=None):
    '''A clean axes with a light grid; returns the axes.'''
    if ax is None:
        _, ax = plt.subplots()
    if title:
        ax.set_title(title, color=DARK)
    return ax


## 1. Zoom into the hole

The function $f(x)=\dfrac{x^2-1}{x-1}$ is *undefined* at $x=1$ — plug in and you get $\tfrac00$. But everywhere else it is **identical** to the line $y=x+1$, because $\dfrac{(x-1)(x+1)}{x-1}=x+1$ whenever $x\neq1$.

Drag the **zoom** slider to shrink the window around $x=1$. The graph becomes indistinguishable from the red line $y=x+1$ — yet the open circle at $(1,2)$ never fills in. The limit is $2$; the *value* there does not exist. The limit sees right past the hole.

In [2]:
def _zoom_hole(half_width=1.0):
    fig, ax = plt.subplots()
    a, L = 1.0, 2.0
    # The line the function secretly is, everywhere except x=1.
    xs = np.linspace(a - half_width, a + half_width, 400)
    ax.plot(xs, xs + 1, color=RED, lw=2.2, zorder=1,
            label=r'$y = x+1$')
    # f(x) = (x^2-1)/(x-1): sample it avoiding x=1 exactly.
    xf = xs[np.abs(xs - a) > half_width * 1e-3]
    ax.plot(xf, (xf**2 - 1) / (xf - 1), color=BLUE, lw=4.5,
            alpha=0.45, zorder=2,
            label=r'$f(x)=\frac{x^2-1}{x-1}$')
    # The persistent hole.
    ax.scatter([a], [L], s=90, facecolors='white',
               edgecolors=BLUE, linewidths=2.0, zorder=5)
    ax.axhline(L, color=GRAY, lw=0.9, ls=':')
    ax.axvline(a, color=GRAY, lw=0.9, ls=':')
    ax.set_xlim(a - half_width, a + half_width)
    ax.set_ylim(L - half_width, L + half_width)
    ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
    ax.set_title('Zooming in on the hole at $(1,2)$', color=DARK)
    ax.legend(loc='upper left', framealpha=0.9)
    plt.show()

interact(_zoom_hole,
         half_width=FloatLogSlider(value=1.0, base=10, min=-3,
                                   max=0, step=0.1,
                                   description='zoom (window)'));


interactive(children=(FloatLogSlider(value=1.0, description='zoom (window)', max=0.0, min=-3.0), Output()), _d…

## 2. The $\varepsilon$–$\delta$ game

Here is the definition the whole subject rests on, made visual for $f(x)=x^2$ at $a=1$, where $L=1$:

> For every output tolerance $\varepsilon>0$ there is an input closeness $\delta>0$ so that $0<|x-a|<\delta$ forces $|f(x)-L|<\varepsilon$.

The skeptic picks $\varepsilon$ (the **horizontal** band $L\pm\varepsilon$). You must answer with a $\delta$ (the **vertical** band $a\pm\delta$) so that the graph stays inside the box. Below, $\delta$ is found *numerically* as the largest value that works. The point above $a$ is drawn hollow — the definition deliberately ignores it.

In [3]:
def _largest_delta(f, a, L, eps, dmax=2.0, n=4000):
    '''Largest delta in (0, dmax] with |f(x)-L|<eps for 0<|x-a|<delta.'''
    # Scan offsets out from a; stop at the first violation on either side.
    offs = np.linspace(dmax / n, dmax, n)
    ok = (np.abs(f(a + offs) - L) < eps) & (np.abs(f(a - offs) - L) < eps)
    bad = np.where(~ok)[0]
    return dmax if bad.size == 0 else offs[bad[0]] - dmax / n

def _eps_delta(eps=0.6):
    fig, ax = plt.subplots()
    a, L = 1.0, 1.0
    f = lambda x: x**2
    delta = _largest_delta(f, a, L, eps)
    xs = np.linspace(a - 1.1, a + 1.1, 500)
    ax.plot(xs, f(xs), color=BLUE, lw=2.4, label=r'$f(x)=x^2$')
    # Output band L +/- eps (horizontal).
    ax.axhspan(L - eps, L + eps, color=SHADE2, alpha=0.55,
               label=r'$|f(x)-L|<\varepsilon$')
    # Input band a +/- delta (vertical).
    ax.axvspan(a - delta, a + delta, color=SHADE, alpha=0.55,
               label=r'$|x-a|<\delta$')
    ax.axhline(L, color=GRAY, lw=0.8, ls=':')
    ax.axvline(a, color=GRAY, lw=0.8, ls=':')
    # Hollow point above a: the definition excludes x=a.
    ax.scatter([a], [L], s=80, facecolors='white',
               edgecolors=DARK, linewidths=1.8, zorder=6)
    ax.annotate(r'$\delta$', xy=(a + delta, L - eps),
                xytext=(a + delta + 0.04, L - eps - 0.55),
                color=BLUE, fontsize=13,
                arrowprops=dict(arrowstyle='->', color=BLUE))
    ax.annotate('', xy=(a, L - eps - 0.35),
                xytext=(a + delta, L - eps - 0.35),
                arrowprops=dict(arrowstyle='<->', color=BLUE, lw=1.5))
    ax.set_xlim(a - 1.1, a + 1.1); ax.set_ylim(-0.2, 2.6)
    ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
    ax.set_title(rf'$\varepsilon={eps:.2f}$  $\Rightarrow$  '
                 rf'working $\delta={delta:.3f}$', color=DARK)
    ax.legend(loc='upper left', framealpha=0.9, fontsize=10)
    plt.show()

interact(_eps_delta,
         eps=FloatSlider(value=0.6, min=0.05, max=1.5, step=0.05,
                         description='\u03b5 (epsilon)'));


interactive(children=(FloatSlider(value=0.6, description='ε (epsilon)', max=1.5, min=0.05, step=0.05), Output(…

## 3. The IVT as an algorithm: bisection

The Intermediate Value Theorem promises that a continuous function which **changes sign** on $[a,b]$ must hit $0$ somewhere between. Here $g(x)=x^3+x-1$ has $g(0)=-1<0$ and $g(1)=1>0$, so a root is trapped in $[0,1]$.

Bisection is the theorem made into a method: check the midpoint, keep whichever half still straddles the sign change, repeat. Slide the **steps** to watch the bracket halve around the root. The red midpoint is the current estimate.

In [4]:
def _bisect(steps=4):
    g = lambda x: x**3 + x - 1
    a, b = 0.0, 1.0
    history = [(a, b)]
    for _ in range(steps):
        m = 0.5 * (a + b)
        if g(a) * g(m) <= 0:
            b = m
        else:
            a = m
        history.append((a, b))
    m = 0.5 * (a + b)

    fig, ax = plt.subplots()
    xs = np.linspace(-0.05, 1.05, 400)
    ax.plot(xs, g(xs), color=BLUE, lw=2.4, label=r'$g(x)=x^3+x-1$')
    ax.axhline(0, color=GRAY, lw=1.0)
    # Current bracketing interval, shaded.
    ax.axvspan(a, b, color=SHADEG, alpha=0.6,
               label=f'bracket  [{a:.4f}, {b:.4f}]')
    # Sign of each endpoint.
    for x0 in (a, b):
        col = GREEN if g(x0) > 0 else RED
        ax.scatter([x0], [g(x0)], s=70, color=col, zorder=5)
        ax.annotate('$+$' if g(x0) > 0 else r'$-$',
                    xy=(x0, g(x0)), xytext=(x0, g(x0) + 0.18),
                    ha='center', color=col, fontsize=14)
    # Current midpoint estimate.
    ax.scatter([m], [g(m)], s=95, color=RED, zorder=6, marker='D',
               label=f'estimate  x \u2248 {m:.5f}')
    ax.axvline(m, color=RED, lw=1.0, ls='--')
    ax.set_xlim(-0.05, 1.05); ax.set_ylim(-1.3, 1.3)
    ax.set_xlabel('$x$'); ax.set_ylabel('$g(x)$')
    ax.set_title(f'Bisection step {steps}:  '
                 f'width {b - a:.5f},  g(x) = {g(m):+.5f}', color=DARK)
    ax.legend(loc='upper left', framealpha=0.9, fontsize=10)
    plt.show()

interact(_bisect,
         steps=IntSlider(value=4, min=0, max=20, step=1,
                         description='steps'));


interactive(children=(IntSlider(value=4, description='steps', max=20), Output()), _dom_classes=('widget-intera…

## 4. Two limits at $0$, and the squeeze

Pick a function from the dropdown and zoom toward $x=0$.

- $\dfrac{\sin x}{x}$ is $\tfrac00$ at the origin, yet it sails straight through height **$1$** with only a hole at $0$ — the most important limit in the book (it becomes a *slope* in Chapter 3).
- $x^2\sin\!\big(\tfrac1x\big)$ wiggles infinitely fast near $0$, but it is trapped between the envelopes $\pm x^2$. Both envelopes head to $0$, so by the **Squeeze Theorem** the function is pinned to $0$ as well.

In [5]:
def _near_zero(which='sin(x)/x', half_width=4.0):
    fig, ax = plt.subplots()
    xs = np.linspace(-half_width, half_width, 2000)
    xs = xs[np.abs(xs) > half_width * 1e-4]  # avoid x=0 exactly
    if which == 'sin(x)/x':
        ax.plot(xs, np.sin(xs) / xs, color=BLUE, lw=2.4,
                label=r'$\frac{\sin x}{x}$')
        ax.axhline(1, color=GREEN, lw=1.2, ls='--',
                   label=r'$y=1$ (the limit)')
        # Hole at (0,1): the value is missing, the limit is 1.
        ax.scatter([0], [1], s=80, facecolors='white',
                   edgecolors=BLUE, linewidths=1.8, zorder=6)
        ax.set_ylim(-0.35, 1.25)
        ax.set_title('$\\sin(x)/x \\to 1$  '
                     '(a hole at $0$, no jump)', color=DARK)
    else:
        ax.plot(xs, xs**2 * np.sin(1 / xs), color=BLUE, lw=1.8,
                label=r'$x^2\sin(1/x)$')
        env = np.linspace(-half_width, half_width, 400)
        ax.plot(env, env**2, color=RED, lw=1.6, ls='--',
                label=r'$+x^2$')
        ax.plot(env, -env**2, color=RED, lw=1.6, ls='--',
                label=r'$-x^2$')
        ax.fill_between(env, -env**2, env**2, color=SHADE2,
                        alpha=0.35)
        ax.scatter([0], [0], s=70, color=GREEN, zorder=6,
                   label='squeezed to $0$')
        ax.set_ylim(-half_width**2 * 1.05, half_width**2 * 1.05)
        ax.set_title('$x^2\\sin(1/x)$ pinned between '
                     '$\\pm x^2$', color=DARK)
    ax.axvline(0, color=GRAY, lw=0.8, ls=':')
    ax.set_xlim(-half_width, half_width)
    ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
    ax.legend(loc='upper right', framealpha=0.9, fontsize=10)
    plt.show()

interact(_near_zero,
         which=Dropdown(options=['sin(x)/x', 'x^2 sin(1/x)'],
                        value='sin(x)/x', description='function'),
         half_width=FloatSlider(value=4.0, min=0.25, max=8.0,
                                step=0.25, description='zoom (window)'));


interactive(children=(Dropdown(description='function', options=('sin(x)/x', 'x^2 sin(1/x)'), value='sin(x)/x')…

---

*Every picture here is the same lesson: the limit is decided by the approach, not the destination. Hold onto that — in Chapter 3 a slope will be defined as a fraction that is exactly $\tfrac00$ at the point of interest, and tamed by precisely this idea.*